# Quantum Lockbox

We will use Q# to generate a quantum random key, then use Python to lock and unlock a short message.

The idea to remember is simple:

`quantum physics -> secret key -> encrypted message`


In [ ]:
%pip install --upgrade "qdk[azure]" ipykernel


In [ ]:
import qsharp

qsharp.init(project_root="..")


In [ ]:
bit_result = qsharp.eval("QuantumLockbox.RandomBit()")
bit_value = 1 if bit_result == qsharp.Result.One else 0

print("Single quantum bit:", bit_value)


In [ ]:
raw_results = qsharp.eval("QuantumLockbox.RandomKey8()")
bits = [1 if result == qsharp.Result.One else 0 for result in raw_results]

print("8 quantum bits:", bits)


In [ ]:
def bits_to_int(bits):
    value = 0
    for bit in bits:
        value = (value << 1) | bit
    return value


def xor_with_key(data, key):
    return bytes(byte ^ key for byte in data)


key = bits_to_int(bits)
message = "Meet at the lockbox after school."
plaintext = message.encode("utf-8")
ciphertext = xor_with_key(plaintext, key)
decrypted = xor_with_key(ciphertext, key).decode("utf-8")

print("Key as int   :", key)
print("Original     :", message)
print("Encrypted    :", ciphertext.hex(" "))
print("Decrypted    :", decrypted)


## Optional Azure Quantum Submission

Use a simulator first. Before running the next cell:

1. Run `az login`
2. Set your workspace with Azure CLI
3. Set `AZURE_QUANTUM_RESOURCE_ID`


In [ ]:
import os

from qdk.azure import Workspace

qsharp.init(project_root="..", target_profile=qsharp.TargetProfile.Base)
program = qsharp.compile("QuantumLockbox.RandomKey8()")

workspace = Workspace(resource_id=os.environ["AZURE_QUANTUM_RESOURCE_ID"])
target_name = os.getenv("AZURE_QUANTUM_TARGET", "rigetti.sim.qvm")
target = workspace.get_targets(target_name)
job = target.submit(program, "quantum-lockbox-random-key", shots=10)

print(job.get_results())


## Challenge

Try the 16-bit extension:

- call `QuantumLockbox.RandomKey16()`
- convert 16 bits into a larger integer
- update the XOR code to use two key bytes instead of one
